# Решение

In [52]:
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GridSearchCV, PredefinedSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone

# Пути к данным.
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR.parent / "data"

TARGET = "target"

# Эти колонки не используем как признаки модели.
ID_COLUMNS = {"lead_id", "user_id"}
TIME_COLUMNS = {"assignment_ts", "assignment_date"}
NON_FEATURE_COLUMNS = ID_COLUMNS | TIME_COLUMNS | {TARGET, "split"}

RANDOM_STATE = 42

## Загрузка данных

Загружаем обучающую выборку, тестовую выборку и события.  
В baseline модель использует только `train.csv` и `test.csv`.

In [32]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
events = pd.read_csv(DATA_DIR / "events.csv")

print("train:", train.shape)
print("test:", test.shape)
print("events:", events.shape)
print("target mean:", train[TARGET].mean())

train: (13694, 119)
test: (4306, 118)
events: (254705, 7)
target mean: 0.20746312253541696


**Преобразуем время в datetime**

In [33]:
train = train.copy()
test = test.copy()

for dateframe in [train, test]:
    dateframe['assignment_date'] = pd.to_datetime(dateframe['assignment_date'], errors="coerce")
    dateframe['assignment_ts'] = pd.to_datetime(dateframe['assignment_ts'], errors="coerce")
    dateframe['assignment_ts_norm'] = dateframe['assignment_ts'].dt.normalize()
events['event_ts'] = pd.to_datetime(events['event_ts'], errors="coerce")

In [34]:
print(list(train.columns))
train.head()

['lead_id', 'user_id', 'assignment_ts', 'assignment_date', 'lead_source', 'call_center', 'region', 'car_segment', 'lead_channel', 'user_tenure_bucket', 'price_bucket', 'assignment_hour', 'assignment_weekday', 'is_weekend', 'user_active_days_30d', 'user_age_days', 'prior_assignments_30d', 'seller_inventory_count', 'seller_response_rate_30d', 'item_price_log', 'car_age_years', 'mileage_km_log', 'item_views_1d', 'item_views_3d', 'item_views_7d', 'item_views_14d', 'item_views_30d', 'item_views_90d', 'item_favorites_1d', 'item_favorites_3d', 'item_favorites_7d', 'item_favorites_14d', 'item_favorites_30d', 'item_favorites_90d', 'detail_expands_1d', 'detail_expands_3d', 'detail_expands_7d', 'detail_expands_14d', 'detail_expands_30d', 'detail_expands_90d', 'photo_swipes_1d', 'photo_swipes_3d', 'photo_swipes_7d', 'photo_swipes_14d', 'photo_swipes_30d', 'photo_swipes_90d', 'seller_page_views_1d', 'seller_page_views_3d', 'seller_page_views_7d', 'seller_page_views_14d', 'seller_page_views_30d', 's

,lead_id,user_id,assignment_ts,assignment_date,lead_source,call_center,region,car_segment,lead_channel,user_tenure_bucket,...,leadgen_prev_positive_30d,leadgen_prev_positive_90d,active_days_auto_1d,active_days_auto_3d,active_days_auto_7d,active_days_auto_14d,active_days_auto_30d,active_days_auto_90d,target,assignment_ts_norm
0,lead_f57db09ab39ae3e7,user_0000001,2026-04-22 11:56:00,2026-04-22,CRM,external,west,budget,retargeting,warm,...,0.0,0.0,0.0,0.0,0.0,4.0,9.0,26.0,0,2026-04-22
1,lead_a6184b8a8165a27b,user_0000002,2026-04-07 14:49:00,2026-04-07,CRM,voxys,north,standard,partner,warm,...,NaN,1.0,0.0,0.0,0.0,NaN,4.0,5.0,0,2026-04-07
2,lead_229c2a117dbac203,user_0000003,2026-04-12 17:01:00,2026-04-12,Perf,external,north,budget,retargeting,new,...,0.0,NaN,2.0,4.0,1.0,10.0,12.0,52.0,0,2026-04-12
3,lead_16b19e58042ef905,user_0000005,2026-04-13 21:39:00,2026-04-13,Model,voxys,east,premium,partner,warm,...,1.0,0.0,0.0,1.0,0.0,3.0,2.0,NaN,1,2026-04-13
4,lead_734c227324978a36,user_0000006,2026-04-12 18:01:00,2026-04-12,CRM,voxys,central,budget,retargeting,warm,...,0.0,0.0,0.0,0.0,0.0,1.0,2.0,9.0,0,2026-04-12


In [35]:
events.head()

,lead_id,user_id,event_ts,event_type,item_price_log,src_slot,ctx_seq
0,lead_00025e9610a0d90d,user_0016636,2026-03-27 06:41:00,chat_open,13.303438,19.0,c02
1,lead_00025e9610a0d90d,user_0016636,2026-03-31 09:10:00,item_view,13.322707,6.0,c06
2,lead_00025e9610a0d90d,user_0016636,2026-04-02 22:04:00,item_view,13.395721,10.0,c06
3,lead_00025e9610a0d90d,user_0016636,2026-04-04 09:19:00,search,13.395955,10.0,c04
4,lead_00025e9610a0d90d,user_0016636,2026-04-07 12:36:00,item_view,13.472769,2.0,c02


## Анализ даных

In [37]:
print("Доля положительного класса:", round(train[TARGET].mean(), 4))

Доля положительного класса: 0.2075


In [38]:
dtype_report = train.dtypes.astype(str).value_counts().rename("columns_count")
display(dtype_report.to_frame())

,columns_count
float64,107
str,9
datetime64[us],3
int64,1


In [39]:
print("Самые частые события:")
display(events["event_type"].value_counts(dropna=False).to_frame("count"))

print("Время:")
print(events["event_ts"].min(), "—", events["event_ts"].max())

print("Доля кол-ва пропусков:")
display(events.isna().mean().sort_values(ascending=False).to_frame("missing_share"))

Самые частые события:


,count
event_type,
item_view,120905
search,61101
favorite,26333
chat_open,24797
call_click,21569


Время:
2026-03-08 10:03:00 — 2026-04-27 20:28:00
Доля кол-ва пропусков:


,missing_share
lead_id,0.0
user_id,0.0
event_ts,0.0
event_type,0.0
item_price_log,0.0
src_slot,0.0
ctx_seq,0.0


## Feature engineering

### Добавление новых признаков из даты 
Тк большинство признаков которые можно извлечь из даты уже есть в датафрейме, я добавила временной тренд и циклическое кодирование часа и дня недели с помощью `sin` и `cos`

In [43]:
reference_date = (pd.to_datetime(train['assignment_date']).min().normalize())

print(f'Исходная дата : {reference_date}\n')


def add_data_features(dataframe, reference_date=reference_date):
    result = dataframe.copy()

    assignment_date = (pd.to_datetime(result['assignment_date']).dt.normalize())

    hour = result["assignment_hour"]
    weekday = result["assignment_weekday"]

    result["days_from_start"] = (assignment_date - reference_date).dt.days.astype("int16")

    result["assignment_hour_sin"] = np.sin(2 * np.pi * hour / 24)
    result["assignment_hour_cos"] = np.cos(2 * np.pi * hour / 24)

    result["assignment_weekday_sin"] = np.sin(2 * np.pi * weekday / 7)
    result["assignment_weekday_cos"] = np.cos(2 * np.pi * weekday / 7)
    
    return result


train = add_data_features(train)

test = add_data_features(test)

Исходная дата : 2026-04-07 00:00:00



### Добавление новых признаков из файла `events.csv`

In [ ]:
## todo

## Предобработка данных

In [44]:
feature_columns = [column
    for column in train.columns
    if column not in NON_FEATURE_COLUMNS
    and column in test.columns
]

numeric_columns = [
    column
    for column in feature_columns
    if pd.api.types.is_numeric_dtype(train[column])
]

categorical_columns = [
    column
    for column in feature_columns
    if column not in numeric_columns
]

print("Числовых признаков:", len(numeric_columns))
print("Категориальных признаков:", len(categorical_columns))
print("Всего признаков до one-hot:", len(feature_columns))
print("Категориальные признаки:", sorted(categorical_columns))

Числовых признаков: 107
Категориальных признаков: 8
Всего признаков до one-hot: 115
Категориальные признаки: ['assignment_ts_norm', 'call_center', 'car_segment', 'lead_channel', 'lead_source', 'price_bucket', 'region', 'user_tenure_bucket']


In [46]:
numeric_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median", add_indicator=True),
        ),
        ("scaler", StandardScaler()),
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                dtype=np.float32,
            ),
        ),
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor, numeric_columns),
        ("cat", categorical_preprocessor, categorical_columns),
    ],
    remainder="drop",
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

## Валидация

Так как тестовая выборка находится позже train по времени, лучше валидироваться на последних датах train.  
Это ближе к реальному сценарию, чем случайный split.

In [50]:
def make_validation_split(train_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Делит train по времени: ранние даты в обучение, поздние даты в валидацию."""
    if "assignment_date" in train_df.columns:
        dates = pd.to_datetime(train_df["assignment_date"]).dt.date
        ordered_dates = sorted(dates.unique())
        cutoff = ordered_dates[int(len(ordered_dates) * 0.8)]

        train_part = train_df[dates < cutoff]
        valid_part = train_df[dates >= cutoff]
        return train_part, valid_part

    return train_test_split(
        train_df,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=train_df[TARGET],
    )


train_part, valid_part = make_validation_split(train)

print("train_part:", train_part.shape)
print("valid_part:", valid_part.shape)

train_part: (10272, 125)
valid_part: (3422, 125)


## Модель Logistic Regression

### Подбор гиперпараметров

In [58]:
logreg_pipeline = Pipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        (
            "model",
            LogisticRegression(
                solver="liblinear",
                max_iter=3000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

search_data = pd.concat(
    [train_part, valid_part],
    ignore_index=True,
)

X_search = search_data[feature_columns]
y_search = search_data[TARGET]

test_fold = np.concatenate(
    [
        np.full(len(train_part), -1, dtype=int),
        np.zeros(len(valid_part), dtype=int),
    ]
)

predefined_split = PredefinedSplit(test_fold=test_fold)

In [60]:
param_grid = {
    "model__C": [
        0.001,
        0.003,
        0.01,
        0.03,
        0.1,
        0.3,
        1.0,
        3.0,
        10.0,
        30.0,
        100.0,
    ],

    "model__l1_ratio": [
        0.0,
        1.0,
    ],

    "model__class_weight": [
        None,
        "balanced",
    ],
}

grid_search = GridSearchCV(
    estimator=logreg_pipeline,
    param_grid=param_grid,
    scoring="average_precision",
    cv=predefined_split,
    n_jobs=-1,
    verbose=1,
    refit=False,
)

grid_search.fit(X_search, y_search)

best_params = grid_search.best_params_

print("Лучшие параметры:", best_params)
print(f"Лучший validation AP: {grid_search.best_score_:.5f}")


Fitting 1 folds for each of 44 candidates, totalling 44 fits
Лучшие параметры: {'model__C': 0.1, 'model__class_weight': None, 'model__l1_ratio': 1.0}
Лучший validation AP: 0.48257


### Финальная модель

In [61]:
best_model = clone(logreg_pipeline).set_params(**best_params)

best_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [63]:
grid_results = (
    pd.DataFrame(grid_search.cv_results_)
    .sort_values("rank_test_score")
)

display(
    grid_results[
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "param_model__C",
            "param_model__l1_ratio",
            "param_model__class_weight",
        ]
    ].head(10)
)

,rank_test_score,mean_test_score,std_test_score,param_model__C,param_model__l1_ratio,param_model__class_weight
17,1,0.482571,0.0,0.10,1.0,NaN
13,2,0.482103,0.0,0.03,1.0,NaN
15,3,0.480506,0.0,0.03,1.0,balanced
21,4,0.480311,0.0,0.30,1.0,NaN
12,5,0.480041,0.0,0.03,0.0,NaN
25,6,0.479604,0.0,1.00,1.0,NaN
29,7,0.479409,0.0,3.00,1.0,NaN
37,8,0.479281,0.0,30.00,1.0,NaN
16,9,0.479275,0.0,0.10,0.0,NaN
33,10,0.479273,0.0,10.00,1.0,NaN


In [64]:
best_model.fit(
    train_part[feature_columns],
    train_part[TARGET],
)

valid_scores = best_model.predict_proba(
    valid_part[feature_columns]
)[:, 1]

valid_ap = average_precision_score(
    valid_part[TARGET],
    valid_scores,
)

print(f"Average Precision: {valid_ap:.5f}")

Average Precision: 0.48257


## Модель CatBoost

## Submission

Обучаем модель на всем train и строим score для test.  
Файл для отправки должен содержать две колонки: `lead_id` и `score`.

In [66]:
model = best_model

In [68]:
# Финально обучаем модель на всей обучающей выборке.
model.fit(train[feature_columns], train[TARGET])

test_scores = model.predict_proba(test[feature_columns])[:, 1]

submission = pd.DataFrame(
    {
        "lead_id": test["lead_id"].astype(str),
        "score": test_scores,
    }
)

submission.to_csv(DATA_DIR / "submission.csv", index=False)
submission.head()

,lead_id,score
0,lead_97e409eb8f8c8246,0.859830
1,lead_55310edb4489f9e9,0.356279
2,lead_e7f653a2c6a7eee8,0.430982
3,lead_22f8e1cfc487ac20,0.111984
4,lead_48b638b839abfac3,0.096295


In [69]:
# Минимальные проверки формата перед загрузкой.
assert list(submission.columns) == ["lead_id", "score"]
assert len(submission) == len(test)
assert submission["lead_id"].is_unique
assert submission["score"].between(0, 1).all()

print("submission.csv is ready")

submission.csv is ready
